# Employee Data — Silver Layer Transformation
### Bronze → Cleansed Dimensions & Facts (Star Schema)

**Purpose:** This notebook reads raw bronze tables and transforms them into a clean, typed, business-ready **star schema** in the silver layer. Invalid rows are filtered out, columns are cast to proper types, audit metadata is stripped, and derived fields (tenure, seniority, promotion eligibility) are computed.

**Transformations Applied:**
- Filter out invalid rows (`_bronze_is_valid = false`)
- Cast IDs to `IntegerType`, dates to `DateType`, monetary amounts to `Decimal(12,2)`
- Derive business attributes: `years_of_service`, `seniority_level`, `promotion_eligible`, `total_compensation`
- Drop bronze audit columns and replace with `_silver_ingested_at`

**Tables Produced:**

| Type | Table | Description |
|------|-------|-------------|
| Dimension | `dim_employee` | Master employee record with name, tenure, seniority, promotion flag |
| Dimension | `dim_date_employee` | Date dimension built from all date columns across the domain |
| Dimension | `dim_district` | School district reference data |
| Dimension | `dim_certification` | Certification code lookup |
| Dimension | `dim_pab_item` | Pay-and-benefit line item descriptions with category |
| Fact | `fact_pay_and_benefits` | Annual compensation summary per employee |
| Fact | `fact_pab_lineitem` | Granular pay/benefit line items with amounts and dates |
| Fact | `fact_teacher_certification` | Teacher certification events with active/expired status |

---
_Upstream: [employee_bronze](#) · Downstream: [employee_gold](#)_

In [0]:
# ── PySpark imports ──
# functions.* → column-level operations (casting, date math, conditionals)
# types → explicit type hints for casting bronze columns to proper data types

from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, StringType, BooleanType, TimestampType

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# Silver layer reads from bronze/ and writes cleansed output to silver/.

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema references ──
bronze_sch = "bronze_employee"
silver_sch = "silver_employee"
gold_sch = "gold_employee"
employee_db = "employeedatacatalog"

In [0]:
# ============================================================
# Load all 13 bronze tables into DataFrames
# ============================================================
# Each table maps to a CSV that was ingested in the bronze layer.
# We read them as Spark tables from the bronze_employee schema.

# Core entity tables
df_employee = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.employee")
df_district = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.district")
df_admin = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.admin")
df_benefit = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.benefit")
df_certification = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.certification")

# Compensation / pay tables
df_ot_pay = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.ot_pay")
df_other_employee = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.other_employee")
df_pab_item = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.pab_item")
df_pab_lineitem = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.pab_lineitem")
df_reg_pay = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.reg_pay")

# Teacher & certification tables
df_teacher = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.teacher")
df_teacher_cert = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.teacher_cert")

# Annual pay-and-benefits totals
df_total_pab = spark.sql("SELECT * FROM employeedatacatalog.bronze_employee.total_pab")

In [0]:
# ============================================================
# DIMENSION: dim_employee
# ============================================================
# The most important dimension in the model. Combines employee
# demographics with derived HR attributes:
#   - years_of_service: months between hire date and today
#   - total_experience: service + prior experience
#   - seniority_level: bucketed from years_of_service
#   - promotion_eligible: true if 5+ yrs service AND 8+ yrs total experience

dim_employee = df_employee.filter(col("_bronze_is_valid") == True)\
                          .withColumn("DISTRICT_ID", col("DISTRICT_ID").cast(IntegerType()))\
                          .withColumn("IS_ADMIN", col("IS_ADMIN").cast(BooleanType()))\
                          .withColumn("IS_TEACHER", col("IS_TEACHER").cast(BooleanType()))\
                          .withColumnRenamed("EMP_FNAME","EMPLOYEE_NAME")\
                          .withColumn("EMPLOYEE_NAME",                    # Merge first + last into full name
                              concat(trim(col("EMPLOYEE_NAME")), lit(" "), trim(col("EMP_LNAME"))))\
                          .withColumn("years_of_service",                 # Tenure from hire date to today
                              floor(months_between(current_date(), col("HIREDATE")) / 12))\
                          .withColumn("total_experience",                 # Service + any prior years
                              col("years_of_service") + coalesce(col("PREVIOUS_EXPERIENCE_YEARS"), lit(0)))\
                          .withColumn("seniority_level",                  # HR seniority classification
                              when(col("years_of_service") >= 20, lit("Senior"))
                              .when(col("years_of_service") >= 10, lit("Mid-Senior"))
                              .when(col("years_of_service") >= 5, lit("Mid-Level"))
                              .when(col("years_of_service") >= 2, lit("Junior"))
                              .otherwise(lit("Entry (<2 yrs)")))\
                          .withColumn("promotion_eligible",               # Business rule: 5yr service + 8yr total
                              when((col("years_of_service") >= 5) & (col("total_experience") >= 8), True)
                              .otherwise(False))\
                          .withColumn("_silver_ingested_at", current_timestamp())\
                          .drop("EMP_LNAME", "DIRECT_ADMIN_ID",           # Drop bronze audit + redundant cols
                                "_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


dim_employee.limit(10).display()                                          # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
dim_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/dim_employee")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.dim_employee
          USING DELTA
          LOCATION '{silver_path}/dim_employee'""")

In [0]:
# ============================================================
# DIMENSION: dim_date_employee
# ============================================================
# A role-playing date dimension built by collecting every distinct
# date that appears anywhere in the employee domain (hire dates,
# cert dates, pay periods, etc.), then enriching with calendar attributes.
#
# This avoids hardcoding a date range and guarantees we only store
# dates that actually appear in our data.

# Step 1 — Collect all unique dates from across the employee domain
df_all_dates = (
    df_employee.select(col("HIREDATE").alias("date"))
    .union(df_admin.select(to_date(col("ADMIN_START_DATE")).alias("date")))
    .union(df_admin.select(to_date(col("ADMIN_END_DATE")).alias("date")))
    .union(df_teacher_cert.select(to_date(col("DATE_EFFECTIVE")).alias("date")))
    .union(df_teacher_cert.select(to_date(col("DATE_EXPIRES")).alias("date")))
    .union(df_pab_lineitem.select(to_date(col("BEG_DATE")).alias("date")))
    .union(df_pab_lineitem.select(to_date(col("END_DATE")).alias("date")))
    .union(df_total_pab.select(to_date(col("DATE_LAST_CALC")).alias("date")))
    .filter(col("date").isNotNull())
    .distinct()
)

# Step 2 — Enrich with calendar attributes
dim_date_employee = df_all_dates \
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("week", weekofyear(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("day_name", date_format(col("date"), "EEEE")) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("is_weekend",                                              # Sat=7, Sun=1
        when((dayofweek(col("date")) == 1) | (dayofweek(col("date")) == 7), True).otherwise(False)) \
    .withColumn("_silver_ingested_at", current_timestamp()) \
    .orderBy("date_key")

dim_date_employee.limit(10).display()                                      # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
dim_date_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/dim_date_employee")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.dim_date_employee
          USING DELTA
          LOCATION '{silver_path}/dim_date_employee'""")

In [0]:
# ============================================================
# DIMENSION: dim_district
# ============================================================
# Clean district reference data. Handles "NULL" string values in
# SUPERINTENDENT_ID by converting them to actual nulls before casting.

dim_district = df_district.filter(col("_bronze_is_valid") == True)\
                          .withColumn("DISTRICT_ID", col("DISTRICT_ID").cast(IntegerType()))\
                          .withColumn("SUPERINTENDENT_ID",               # Handle string "NULL" → actual null
                              when(col("SUPERINTENDENT_ID").isin("NULL", ""), lit(None))
                              .otherwise(col("SUPERINTENDENT_ID")).cast(IntegerType()))\
                          .withColumn("_silver_ingested_at", current_timestamp())\
                          .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


dim_district.limit(10).display()                                         # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
dim_district.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_district")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.dim_district
          USING DELTA
          LOCATION '{silver_path}/dim_district'""")

In [0]:
# ============================================================
# DIMENSION: dim_certification
# ============================================================
# Simple lookup table mapping CERT_ID → STATE_CERT_CODE + description.
# Minimal transformation needed — just type casting and audit swap.

dim_certification = df_certification.filter(col("_bronze_is_valid") == True)\
                                    .withColumn("CERT_ID", col("CERT_ID").cast(IntegerType()))\
                                    .withColumn("_silver_ingested_at", current_timestamp())\
                                    .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


dim_certification.limit(10).display()                                    # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
dim_certification.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_certification")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.dim_certification
          USING DELTA
          LOCATION '{silver_path}/dim_certification'""")

In [0]:
# ============================================================
# DIMENSION: dim_pab_item
# ============================================================
# The trickiest dimension — we need to merge three separate bronze
# tables (OT_Pay, Benefit, REG_pay) into a single item lookup.
#
# Strategy:
#   1. Clean each sub-table and rename conflicting columns
#   2. Left-join all three onto pab_item by PAB_ITEM_ID
#   3. Derive item_category from which sub-table had a match:
#      • Has COLLECTIVE_BARGAINING_SECT → "Regular Pay"
#      • Has OT_TAXABLE_CODE_ID → "Overtime Pay"
#      • Has TAXABLE_CODE_ID → "Benefit"
#      • None of the above → "Other"

# Step 1 — Clean sub-tables (filter valid rows, rename conflicting columns)
df_ot_pay_clean = df_ot_pay.filter(col("_bronze_is_valid") == True)\
                           .select(col("PAB_ITEM_ID"),
                                   col("TAXABLE_CODE_ID").alias("OT_TAXABLE_CODE_ID"))  # Rename to avoid collision

df_benefit_clean = df_benefit.filter(col("_bronze_is_valid") == True)\
                             .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")

df_reg_pay_clean = df_reg_pay.filter(col("_bronze_is_valid") == True)\
                             .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")

# Step 2 — Join sub-tables onto the base pab_item dimension
dim_pab_item = df_pab_item.filter(col("_bronze_is_valid") == True)\
                          .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")\
                          .join(df_benefit_clean, "PAB_ITEM_ID", "left")\
                          .join(df_reg_pay_clean, "PAB_ITEM_ID", "left")\
                          .join(df_ot_pay_clean, "PAB_ITEM_ID", "left")\
                          .withColumn("item_category",                   # Derive category from join presence
                              when(col("COLLECTIVE_BARGAINING_SECT").isNotNull(), lit("Regular Pay"))
                              .when(col("OT_TAXABLE_CODE_ID").isNotNull(), lit("Overtime Pay"))
                              .when(col("TAXABLE_CODE_ID").isNotNull(), lit("Benefit"))
                              .otherwise(lit("Other")))\
                          .withColumn("taxable_code",                    # Unify the two taxable code columns
                              coalesce(col("TAXABLE_CODE_ID"), col("OT_TAXABLE_CODE_ID")))\
                          .withColumn("_silver_ingested_at", current_timestamp())\
                          .select("PAB_ITEM_ID", "TYPE", "ITEM_DESC", "item_category",
                                  "taxable_code", "COLLECTIVE_BARGAINING_SECT", "_silver_ingested_at")


dim_pab_item.limit(10).display()                                         # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
dim_pab_item.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_pab_item")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.dim_pab_item
          USING DELTA
          LOCATION '{silver_path}/dim_pab_item'""")

In [0]:
# ============================================================
# FACT: fact_pay_and_benefits
# ============================================================
# One row per employee per tax year. Contains the four major pay
# components (reg, overtime, other, benefits) and a derived
# total_compensation column. This is the primary financial fact
# table used in gold-layer aggregations.
#
# Key transformations:
#   - Cast IDs to IntegerType, monetary columns to Decimal(12,2)
#   - Compute total_compensation = REG + OT + OTHER + BENEFITS (coalesced to 0)
#   - Add date_key for joining to dim_date

fact_pay_and_benefits = df_total_pab.filter(col("_bronze_is_valid") == True)\
                                    .withColumn("EMP_ID", col("EMP_ID").cast(IntegerType()))\
                                    .withColumn("TAX_YEAR", col("TAX_YEAR").cast(IntegerType()))\
                                    .withColumn("PAB_ID", col("PAB_ID").cast(IntegerType()))\
                                    .withColumn("REVIEW_ADMIN_ID", col("REVIEW_ADMIN_ID").cast(IntegerType()))\
                                    .withColumn("REG_PAY", col("REG_PAY").cast("decimal(12,2)"))\
                                    .withColumn("OVERTIME_PAY", col("OVERTIME_PAY").cast("decimal(12,2)"))\
                                    .withColumn("OTHER_PAY", col("OTHER_PAY").cast("decimal(12,2)"))\
                                    .withColumn("TOTAL_BENEFITS", col("TOTAL_BENEFITS").cast("decimal(12,2)"))\
                                    .withColumn("total_compensation",    # Sum of all four pay components
                                        coalesce(col("REG_PAY"), lit(0)) +
                                        coalesce(col("OVERTIME_PAY"), lit(0)) +
                                        coalesce(col("OTHER_PAY"), lit(0)) +
                                        coalesce(col("TOTAL_BENEFITS"), lit(0)))\
                                    .withColumn("DATE_LAST_CALC", to_date(col("DATE_LAST_CALC")))\
                                    .withColumn("date_key",              # FK to dim_date
                                        date_format(col("DATE_LAST_CALC"), "yyyyMMdd").cast(IntegerType()))\
                                    .withColumn("_silver_ingested_at", current_timestamp())\
                                    .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


fact_pay_and_benefits.limit(10).display()                                # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
fact_pay_and_benefits.write.format("delta").mode("overwrite").save(f"{silver_path}/fact_pay_and_benefits")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.fact_pay_and_benefits
          USING DELTA
          LOCATION '{silver_path}/fact_pay_and_benefits'""")

In [0]:
# ============================================================
# FACT: fact_pab_lineitem
# ============================================================
# The most granular financial table — individual pay/benefit line
# items with amounts, date ranges, and posting timestamps.
# Each row links to:
#   - fact_pay_and_benefits via PAB_ID
#   - dim_pab_item via PAB_ITEM_ID
#   - dim_date via beg_date_key / end_date_key

fact_pab_lineitem = df_pab_lineitem.filter(col("_bronze_is_valid") == True)\
                                  .withColumn("PAB_ID", col("PAB_ID").cast(IntegerType()))\
                                  .withColumn("PAB_ITEM_ID", col("PAB_ITEM_ID").cast(IntegerType()))\
                                  .withColumn("AMOUNT_POSTED", col("AMOUNT_POSTED").cast("decimal(12,2)"))\
                                  .withColumn("BEG_DATE", to_date(col("BEG_DATE")))\
                                  .withColumn("END_DATE", to_date(col("END_DATE")))\
                                  .withColumn("POSTED_TIMESTAMP", to_timestamp(col("POSTED_TIMESTAMP")))\
                                  .withColumn("beg_date_key",            # FK to dim_date (period start)
                                      date_format(col("BEG_DATE"), "yyyyMMdd").cast(IntegerType()))\
                                  .withColumn("end_date_key",            # FK to dim_date (period end)
                                      date_format(col("END_DATE"), "yyyyMMdd").cast(IntegerType()))\
                                  .withColumn("_silver_ingested_at", current_timestamp())\
                                  .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


fact_pab_lineitem.limit(10).display()                                    # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
fact_pab_lineitem.write.format("delta").mode("overwrite").save(f"{silver_path}/fact_pab_lineitem")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.fact_pab_lineitem
          USING DELTA
          LOCATION '{silver_path}/fact_pab_lineitem'""")

In [0]:
# ============================================================
# FACT: fact_teacher_certification
# ============================================================
# Links teachers (T_EMP_ID) to their certifications (CERT_ID)
# with effective/expiry dates. Derived columns:
#   - is_active: true if the cert hasn't expired yet
#   - cert_duration_days: days between effective and expiry dates
# Used downstream in gold for compliance and breadth analysis.

fact_teacher_certification = df_teacher_cert.filter(col("_bronze_is_valid") == True)\
                                            .withColumn("T_EMP_ID", col("T_EMP_ID").cast(IntegerType()))\
                                            .withColumn("CERT_ID", col("CERT_ID").cast(IntegerType()))\
                                            .withColumn("DATE_EFFECTIVE", to_date(col("DATE_EFFECTIVE")))\
                                            .withColumn("DATE_EXPIRES", to_date(col("DATE_EXPIRES")))\
                                            .withColumn("is_active",              # Still valid as of today?
                                                when(col("DATE_EXPIRES") >= current_date(), True).otherwise(False))\
                                            .withColumn("cert_duration_days",     # Total cert validity window
                                                datediff(col("DATE_EXPIRES"), col("DATE_EFFECTIVE")))\
                                            .withColumn("_silver_ingested_at", current_timestamp())\
                                            .drop("_bronze_ingested_at", "_bronze_source_file", "_bronze_batch_id", "_bronze_is_valid")


fact_teacher_certification.limit(10).display()                            # Preview only 10 rows to conserve resources

# Persist to silver Delta and register in UC
fact_teacher_certification.write.format("delta").mode("overwrite").save(f"{silver_path}/fact_teacher_certification")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{silver_sch}.fact_teacher_certification
          USING DELTA
          LOCATION '{silver_path}/fact_teacher_certification'""")